Test categorizzazioni controlli

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import re

# === Load ===
df = pd.read_csv("nis2_classified_mpnet.csv")  # <-- percorso nel tuo ambiente

# === Sanity ===
expected = {"Req ID","Cleaned Requirement","Category","Technical_Score","NonTechnical_Score"}
missing = expected - set(df.columns)
if missing:
    raise ValueError(f"Mancano colonne nel CSV: {missing}")

# === Derivati ===
df["delta"] = df["Technical_Score"] - df["NonTechnical_Score"]
df["abs_delta"] = df["delta"].abs()
df["pred_label_sign"] = np.where(df["delta"] >= 0, "technical", "non-technical")

# === Metriche base ===
label_counts = df["Category"].value_counts()
label_perc = (label_counts / len(df)).rename("percentage")

margin_mean = df["abs_delta"].mean()
margin_std  = df["abs_delta"].std()

eps = 0.03  # bordo: separazione piccola
borderline_perc = (df["abs_delta"] < eps).mean()

agreement_sign = (df["pred_label_sign"] == df["Category"]).mean()

# Curva sensibilità alla soglia
taus = np.linspace(-0.10, 0.10, 41)
perc_tech = []
for t in taus:
    pred = np.where(df["delta"] >= t, "technical", "non-technical")
    perc_tech.append((pred == "technical").mean())
sens_df = pd.DataFrame({"tau": taus, "perc_technical": perc_tech})

# Proxy keyword ultra-semplice
tech_kw = {
    "encrypt","encryption","key","keys","certificate","tls","ssl","vpn","firewall",
    "access control","privilege","identity","authentication","authorization","token",
    "patch","vulnerability","hardening","endpoint","malware","antivirus","backup",
    "log","logging","audit","monitor","monitoring","incident","response","siem",
    "network","ip","port","segmentation","mfa","multi-factor","hash","salt","iam"
}
def keyword_proxy(text: str)->str:
    t = (text or "").lower()
    hit = any(kw in t for kw in tech_kw)
    return "technical" if hit else "non-technical"

df["proxy_label"] = df["Cleaned Requirement"].apply(keyword_proxy)
agreement_proxy = (df["proxy_label"] == df["Category"]).mean()

# Piccolo stress test di stabilità (±0.01)
pred_orig = df["Category"]
pred_tight = np.where(df["delta"] >= 0.01, "technical",
                      np.where(df["delta"] <= -0.01, "non-technical", "borderline"))
flip_rate = (pred_tight != pred_orig).mean()

# === Output su disco ===
out_dir = Path("test_results_phase2"); img_dir = out_dir/"images"
img_dir.mkdir(parents=True, exist_ok=True)

summary = pd.DataFrame({
    "metric": [
        "n_items","perc_technical","perc_non_technical",
        "margin_mean_abs_delta","margin_std_abs_delta",
        "perc_borderline_(|Δ|<0.03)","agreement_sign_vs_label",
        "agreement_keyword_proxy","flip_rate_(±0.01_threshold)"
    ],
    "value": [
        len(df),
        (label_perc.get("technical",0.0)),
        (label_perc.get("non-technical",0.0)),
        margin_mean, margin_std, borderline_perc,
        agreement_sign, agreement_proxy, flip_rate
    ]
})
summary.to_csv(out_dir/"classification_summary_metrics.csv", index=False)
df.to_csv(out_dir/"classification_detailed_with_deltas.csv", index=False)

# === Figure 1: Istogramma Δ ===
fig1, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.hist(df["delta"], bins=20, edgecolor="white")
ax1.set_title("Distribuzione di Δ = Technical_Score − NonTechnical_Score")
ax1.set_xlabel("Δ"); ax1.set_ylabel("Frequenza")
ax1.grid(axis="y", linestyle=":", linewidth=0.5)
fig1.tight_layout(); fig1.savefig(img_dir/"hist_delta.png", dpi=200, bbox_inches="tight"); plt.close(fig1)

# === Figure 2: % Technical vs τ ===
fig2, ax2 = plt.subplots(figsize=(8, 4.5))
ax2.plot(sens_df["tau"], sens_df["perc_technical"], marker="o")
ax2.set_title("Sensibilità alla soglia: % Technical vs τ")
ax2.set_xlabel("τ (soglia su Δ)"); ax2.set_ylabel("% Technical")
ax2.grid(True, linestyle=":", linewidth=0.5)
fig2.tight_layout(); fig2.savefig(img_dir/"perc_technical_vs_tau.png", dpi=200, bbox_inches="tight"); plt.close(fig2)

print("DONE. Files in:", out_dir.resolve())

DONE. Files in: /content/test_results_phase2


Test Matching NIS2 NIST

In [ ]:
# === NIS2–NIST Matching: metriche + grafici ===
# Input atteso: /mnt/data/nis2_nist_semantic_matches.csv con colonne:
# - "Req ID"                : identificativo articolo NIS2
# - "Cleaned Requirement"   : testo normalizzato dell'articolo
# - "Matched Controls"      : stringa Python-literal di lista di tuple [(control_id, similarity), ...]
# - "Matched Families"      : stringa Python-literal di lista di famiglie NIST (es. ["AC","AU",...])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast
from pathlib import Path
from collections import Counter

# -------------------------
# Parametri
# -------------------------
INPUT_CSV   = "nis2_nist_semantic_matches.csv"   # cambia percorso se necessario
OUT_DIR     = "test_results_matching"            # cartella di output (CSV + immagini)
THRESH_MIN  = 0.50
THRESH_MAX  = 0.85
THRESH_STEPS= 15
TOP_FAM_N   = 12

# -------------------------
# Load & parsing sicuro
# -------------------------
df = pd.read_csv(INPUT_CSV)

def parse_controls(s):
    """Parsa 'Matched Controls' in lista di tuple (control_id:str, score:float)."""
    try:
        lst = ast.literal_eval(s)
    except Exception:
        return []
    out = []
    for item in lst:
        if isinstance(item, (list, tuple)) and len(item) >= 2:
            cid = str(item[0])
            try:
                score = float(item[1])
            except Exception:
                score = np.nan
            out.append((cid, score))
    return out

def parse_families(s):
    """Parsa 'Matched Families' in lista di stringhe."""
    try:
        lst = ast.literal_eval(s)
        return [str(x) for x in lst]
    except Exception:
        return []

df["controls"] = df["Matched Controls"].apply(parse_controls)
df["families"]  = df["Matched Families"].apply(parse_families)

# -------------------------
# Long form (coppie articolo-controllo)
# -------------------------
rows = []
for _, r in df.iterrows():
    nis2 = r["Req ID"]
    text = r["Cleaned Requirement"]
    for cid, score in r["controls"]:
        rows.append({"Req ID": nis2, "Cleaned Requirement": text, "Control ID": cid, "Similarity": score})
long = pd.DataFrame(rows)

# -------------------------
# Output dirs
# -------------------------
out_dir = Path(OUT_DIR)
img_dir = out_dir / "images"
out_dir.mkdir(parents=True, exist_ok=True)
img_dir.mkdir(parents=True, exist_ok=True)

# -------------------------
# METRICHE
# -------------------------

# 1) Copertura & #match per articolo
df["n_matches"] = df["controls"].apply(lambda L: sum(1 for _, s in L if pd.notna(s)))
coverage = (df["n_matches"] > 0).mean()
summary_articles = pd.DataFrame([{
    "n_articles": len(df),
    "coverage_(>=1_match)": coverage,
    "matches_min": df["n_matches"].min(),
    "matches_max": df["n_matches"].max(),
    "matches_mean": df["n_matches"].mean(),
    "matches_median": df["n_matches"].median(),
}])
summary_articles.to_csv(out_dir / "matching_summary_articles.csv", index=False)

# 2) Statistiche delle similarità (tutte le coppie)
sim = long["Similarity"].dropna()
similarity_stats = pd.DataFrame([{
    "similarity_count": int(sim.size),
    "similarity_mean": sim.mean(),
    "similarity_std":  sim.std(ddof=1),
    "similarity_p25":  sim.quantile(0.25),
    "similarity_p50":  sim.quantile(0.50),
    "similarity_p75":  sim.quantile(0.75),
    "similarity_max":  sim.max(),
}])
similarity_stats.to_csv(out_dir / "matching_similarity_stats.csv", index=False)

# 3) Sweep soglia (robustezza): #match, % articoli coperti, media match/articolo
thresholds = np.linspace(THRESH_MIN, THRESH_MAX, THRESH_STEPS)
sweep_rows = []
for t in thresholds:
    long_t = long[long["Similarity"] >= t]
    total_matches_t = len(long_t)
    nis2_with_any = long_t["Req ID"].nunique()
    frac_articles_with_any = nis2_with_any / len(df) if len(df) else 0.0
    matches_per_art = long_t.groupby("Req ID").size() if len(long_t) else pd.Series(dtype=int)
    mean_matches_per_art = matches_per_art.mean() if not matches_per_art.empty else 0.0
    sweep_rows.append({
        "threshold": round(float(t), 3),
        "total_matches": int(total_matches_t),
        "frac_articles_with_>=1": round(float(frac_articles_with_any), 4),
        "mean_matches_per_article": round(float(mean_matches_per_art), 4),
    })
sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv(out_dir / "matching_threshold_sweep.csv", index=False)

# 4) Top articoli per #match
top_articles = df[["Req ID", "n_matches"]].sort_values("n_matches", ascending=False)
top_articles.to_csv(out_dir / "matching_top_articles.csv", index=False)

# 5) Distribuzione famiglie NIST
fam_counter = Counter()
for fams in df["families"]:
    fam_counter.update(fams)
fam_df = pd.DataFrame(fam_counter.items(), columns=["NIST Family", "Count"]).sort_values("Count", ascending=False)
fam_df.to_csv(out_dir / "matching_families_distribution.csv", index=False)

# 6) Ambiguità del miglior match (gap top1–top2)
def top_gap(L):
    scores = sorted([s for _, s in L if pd.notna(s)], reverse=True)
    if len(scores) == 0:
        return np.nan
    if len(scores) == 1:
        return scores[0]  # unico match: gap “massimo” = top1
    return scores[0] - scores[1]

df["top1_top2_gap"] = df["controls"].apply(top_gap)
df.to_csv(out_dir / "matching_articles_enriched.csv", index=False)
long.to_csv(out_dir / "matching_long_pairs.csv", index=False)

# -------------------------
# GRAFICI (matplotlib, no seaborn)
# -------------------------

# A) Istogramma di tutte le similarità
plt.figure(figsize=(8, 4.5))
plt.hist(sim, bins=40, edgecolor="white")
plt.title("Distribuzione delle similarità coseno (tutte le coppie NIS2–NIST)")
plt.xlabel("Similarità coseno"); plt.ylabel("Frequenza")
plt.grid(axis="y", linestyle=":", linewidth=0.5)
plt.tight_layout()
plt.savefig(img_dir / "hist_all_cosine_similarities.png", dpi=200, bbox_inches="tight")
plt.close()

# B) Barre: #match per articolo NIS2
plt.figure(figsize=(9, 4.5))
order = df.sort_values("n_matches", ascending=False)
plt.bar(order["Req ID"], order["n_matches"])
plt.title("Numero di controlli NIST mappati per articolo NIS2")
plt.xlabel("Articolo NIS2"); plt.ylabel("# Match")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(img_dir / "bar_matches_per_article.png", dpi=200, bbox_inches="tight")
plt.close()

# C) Curva robustezza: #match totali e frazione articoli coperti vs soglia
fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(sweep_df["threshold"], sweep_df["total_matches"], marker="o")
ax1.set_xlabel("Soglia di similarità")
ax1.set_ylabel("Numero di match (>= soglia)")
ax1.grid(True, linestyle=":", linewidth=0.5)
ax2 = ax1.twinx()
ax2.plot(sweep_df["threshold"], sweep_df["frac_articles_with_>=1"], marker="s", linestyle="--")
ax2.set_ylabel("Frazione articoli con ≥1 match")
fig.suptitle("Robustezza al variare della soglia di similarità")
fig.tight_layout()
fig.savefig(img_dir / "threshold_sweep.png", dpi=200, bbox_inches="tight")
plt.close(fig)

# D) Barre: distribuzione famiglie NIST (top N)
top_fam = fam_df.head(TOP_FAM_N)
plt.figure(figsize=(9, 4.5))
plt.bar(top_fam["NIST Family"], top_fam["Count"])
plt.title("Distribuzione delle famiglie NIST nei match")
plt.xlabel("Famiglia NIST"); plt.ylabel("Conteggio")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(img_dir / "bar_family_distribution.png", dpi=200, bbox_inches="tight")
plt.close()

# E) Istogramma del gap (top1 − top2)
valid_gaps = df["top1_top2_gap"].dropna()
plt.figure(figsize=(8, 4.5))
plt.hist(valid_gaps, bins=30, edgecolor="white")
plt.title("Distribuzione del margine tra primo e secondo match (top1 − top2)")
plt.xlabel("Gap di similarità"); plt.ylabel("Frequenza")
plt.grid(axis="y", linestyle=":", linewidth=0.5)
plt.tight_layout()
plt.savefig(img_dir / "hist_top1_top2_gap.png", dpi=200, bbox_inches="tight")
plt.close()

print(f"Done. Risultati salvati in: {out_dir.resolve()}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Carica il CSV
df = pd.read_csv("nis2_nist_semantic_matches_080.csv")

# 1. Distribuzione delle similarità
plt.figure(figsize=(8,5))
sns.histplot(df["similarity"], bins=30, kde=True, color="steelblue")
plt.title("Distribuzione delle similarità coseno")
plt.xlabel("Punteggio di similarità")
plt.ylabel("Numero di match")
plt.tight_layout()
plt.savefig("immagini/similarity_distribution.png", dpi=300)
plt.show()

# 2. Numero di match per articolo NIS2
plt.figure(figsize=(10,6))
df["nis2_id"].value_counts().sort_values().plot(kind="barh", color="slateblue")
plt.title("Numero di controlli NIST associati per articolo NIS2")
plt.xlabel("Numero di match")
plt.ylabel("Articolo NIS2")
plt.tight_layout()
plt.savefig("immagini/matches_per_article.png", dpi=300)
plt.show()

# 3. Numero di match per famiglia NIST
plt.figure(figsize=(10,6))
df["nist_family"].value_counts().sort_values().plot(kind="barh", color="teal")
plt.title("Distribuzione dei match per famiglia NIST")
plt.xlabel("Numero di match")
plt.ylabel("Famiglia NIST")
plt.tight_layout()
plt.savefig("immagini/matches_per_family.png", dpi=300)
plt.show()